In [2]:
import pandas as pd
from pyspark import SparkConf
from pyspark.sql import SparkSession, functions as f, Window


In [3]:
parameters = {
    "spark.driver.maxResultSize": "3g",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.execution.arrow.pyspark.enabled": True,

    # https://docs.kedro.org/en/stable/integrations/pyspark_integration.html#tips-for-maximising-concurrency-using-threadrunner
    "spark.scheduler.mode": "FAIR",
    "spark.driver.extraJavaOptions": "-Djava.security.manager=allow",
    "spark.executor.extraJavaOptions": "-Djava.security.manager=allow",

    "spark.sql.legacy.parquet.nanosAsLong": True,
    "spark.sql.legacy.timeParserPolicy": "LEGACY",
    "spark.driver.memory": "40g",
}

spark_conf = SparkConf().setAll(parameters.items())

In [4]:
spark = SparkSession.builder.appName('New').enableHiveSupport().config(conf=spark_conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/01 13:37:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/06/01 13:37:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/06/01 13:37:33 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [5]:
int_transactions = spark.read.parquet("../data/02_intermediate/int_transactions")
prm_spine = spark.read.parquet("../data/03_primary/prm_spine")

In [6]:
transactions_data = int_transactions.withColumn(
    "_key",
    # f.col("customer_id")
    f.concat_ws("__", "customer_id", "customer_vehicle_id")
).withColumn(
    "_observ_end_dt",
    f.last_day(f.col("transaction_dt"))
    # f.to_date(f.date_trunc("quarter", f.col("transaction_dt")))
)

In [6]:
transactions_data.select("transaction_dt").show(4)

+-------------------+
|     transaction_dt|
+-------------------+
|2023-01-29 01:36:40|
|2023-01-29 01:36:40|
|2021-03-25 17:02:18|
|2021-03-25 17:02:18|
+-------------------+
only showing top 4 rows



In [7]:
transactions_data.select(f.countDistinct("_key")).show()
prm_spine.select(f.countDistinct("_key")).show()
prm_spine.select("_key").distinct().join(
    transactions_data.select("_key").distinct(),
    on="_key",
    how="inner"
).select(f.countDistinct("_key")).show()

+--------------------+
|count(DISTINCT _key)|
+--------------------+
|             3476487|
+--------------------+



+--------------------+
|count(DISTINCT _key)|
+--------------------+
|             3421097|
+--------------------+



+--------------------+
|count(DISTINCT _key)|
+--------------------+
|             3167612|
+--------------------+



In [8]:
transactions_data.select(f.min("_observ_end_dt"), f.max("_observ_end_dt")).show()
prm_spine.select(f.min("_observ_end_dt"), f.max("_observ_end_dt")).show()

+-------------------+-------------------+
|min(_observ_end_dt)|max(_observ_end_dt)|
+-------------------+-------------------+
|         2020-07-31|         2025-12-31|
+-------------------+-------------------+



+-------------------+-------------------+
|min(_observ_end_dt)|max(_observ_end_dt)|
+-------------------+-------------------+
|         2020-07-31|         2025-06-30|
+-------------------+-------------------+



In [7]:
base_sales = prm_spine.join(
    transactions_data,
    on=["_key", "_observ_end_dt"],
    how="left"
)

In [ ]:
print(prm_spine.count())
print(transactions_data.count())
print(base_sales.count())
print(base_sales.dropna(subset="transaction_id").count())
print(base_sales.select("_key", "_observ_end_dt").distinct().count())

132749229
46064056


160057643


40439606


TypeError: DataFrame.distinct() takes 1 positional argument but 3 were given

In [8]:
prm_spine.show(5)

+--------------------+--------------------+--------------+
|                 _id|                _key|_observ_end_dt|
+--------------------+--------------------+--------------+
|6087NKJ__96659893...|E0470497-EF51-4B1...|    2023-08-31|
|6087NKJ__96659893...|E0470497-EF51-4B1...|    2023-09-30|
|6087NKJ__96659893...|E0470497-EF51-4B1...|    2023-10-31|
|6087NKJ__96659893...|E0470497-EF51-4B1...|    2023-11-30|
|6087NKJ__96659893...|E0470497-EF51-4B1...|    2023-12-31|
+--------------------+--------------------+--------------+
only showing top 5 rows



In [9]:
prm_spine.select(f.countDistinct("_id"), f.countDistinct("_observ_end_dt")).show()

25/06/01 13:37:52 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+-------------------+------------------------------+
|count(DISTINCT _id)|count(DISTINCT _observ_end_dt)|
+-------------------+------------------------------+
|            3101979|                            60|
+-------------------+------------------------------+



In [13]:
base_sales.filter(
    f.col("transaction_id").isNotNull()
).show(5, truncate=False)

25/06/01 11:38:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------------------------------------------------------------+--------------+---------------------+------------------------------------+------------------------------------+------------------------------------+------------------------------------+-------------------+-------------------------+-----------------+--------------------------------------------+--------------------------------------------------------------------------------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+------------+------------+---------------------+------------+-------------+------------------+
|_key                                                                      |_observ_end_dt|_id                  |transaction_id                      |customer_id                         |customer_vehicle_id                 |branch_id                           |transaction_dt     |product_family           |product_categ

# Global Sales Transactional Features

In [10]:
base_sales.filter(
    f.col("discount_amount") > 0
).show(5, truncate=False)

25/06/01 13:37:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------------------------------------------------------------+--------------+---------------------+------------------------------------+------------------------------------+------------------------------------+------------------------------------+-------------------+--------------------------+-------------------------+---------------------------------+-----------------------+----------------------------------------------------------------------------------------------------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+--------------------+------------+------------+---------------------+------------+-------------+------------------+
|_key                                                                      |_observ_end_dt|_id                  |transaction_id                      |customer_id                         |customer_vehicle_id                 |branch_id                        

In [16]:
base_sales.filter(
    f.col("product_id") == "CABIN AIR FILTER SERVICES | HYUNDAI-CABIN AIR FILTERS | CABIN AIR FILTER | HYUN 97133-3SAA0"
).show(20, truncate=False)

+--------------------------------------------------------------------------+--------------+---------------------+------------------------------------+------------------------------------+------------------------------------+------------------------------------+-------------------+-------------------------+-------------------------+--------------------+----------------+-------------------------------------------------------------------------------------------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+--------------------+------------+------------+---------------------+------------+-------------+------------------+-------------------+
|_key                                                                      |_observ_end_dt|_id                  |transaction_id                      |customer_id                         |customer_vehicle_id                 |branch_id                           |transa

In [17]:
base_sales.groupBy(
    "product_code"
).agg(
    f.count_distinct("product_family"),
    f.count_distinct("product_category"),
    f.count_distinct("product_sub_category"),
    f.count_distinct("product_id"),
).show(20, truncate=False)

+-----------------------------+------------------------------+--------------------------------+------------------------------------+--------------------------+
|product_code                 |count(DISTINCT product_family)|count(DISTINCT product_category)|count(DISTINCT product_sub_category)|count(DISTINCT product_id)|
+-----------------------------+------------------------------+--------------------------------+------------------------------------+--------------------------+
|ACA-90915YZZD2               |5                             |1                               |1                                   |5                         |
|TOYO 90118-WC341             |5                             |1                               |1                                   |5                         |
|ELECTROMIN-N50               |11                            |1                               |1                                   |11                        |
|HYDRAULIC68                  |20       

In [23]:
base_sales.groupBy(
    "product_code"
).agg(
    f.count_distinct("transaction_id").alias("count")
).orderBy(
    f.desc("count")
).show(50, truncate=False)

+------------------------+-------+
|product_code            |count  |
+------------------------+-------+
|BULK10W30               |4388761|
|PET A-1 SYN             |2379695|
|PETROMINDOT4            |1350065|
|M4612                   |1289503|
|PET A-1 SYN 5W30 - 6X4  |1179304|
|PET10W30                |1048775|
|M4455                   |1008020|
|26300-35503             |672582 |
|M4477                   |621941 |
|PET20W50                |608135 |
|PETGRNCOOLANT           |606274 |
|PETREDCOOLANT           |604185 |
|PETFLEET15W40 -24X1     |603088 |
|BILSTEIN                |523960 |
|TR120 SYNTHETIC ATF OIL |452077 |
|M3600                   |437790 |
|PET A-1 SYN 5W30 1L     |410624 |
|PET5W20HYBRID           |373686 |
|R134A REFRIGERANT GM NEW|356452 |
|M500                    |334603 |
|PETFLEET15W40 - 6X4     |331518 |
|TOY20W50                |331190 |
|TOYO 90430-12031        |315209 |
|PETAIRFRESH             |311672 |
|M3614                   |308945 |
|15208-31U01B       

In [24]:
base_sales.groupBy(
    "product_category"
).agg(
    f.count_distinct("transaction_id").alias("count")
).orderBy(
    f.desc("count")
).show(50, truncate=False)

base_sales.groupBy(
    "product_sub_category"
).agg(
    f.count_distinct("transaction_id").alias("count")
).orderBy(
    f.desc("count")
).show(50, truncate=False)

+--------------------------+-------+
|product_category          |count  |
+--------------------------+-------+
|OIL FILTERS               |8522200|
|OIL                       |8068135|
|OIL SYNTHETIC             |4472570|
|BRAKE FLUID               |1350065|
|TOYOTA-OIL FILTERS        |1253156|
|COOLANTS                  |1209597|
|AIR FILTERS               |860432 |
|BATTERIES                 |775170 |
|HYUNDAI-OIL FILTERS       |686986 |
|CABIN AIR FILTERS         |606407 |
|CONSUMABLES               |542548 |
|ENGINE FLUSH              |533589 |
|TRANSMISSION FLUID        |495706 |
|NORMAL-AC SERVICE         |359031 |
|DRAIN PLUGS               |327298 |
|ADDITIVES                 |322309 |
|FUEL INJECTION            |276795 |
|NISSAN-OIL FILTERS        |244422 |
|NISSAN-MISCELLANEOUS      |196349 |
|WIPER BLADES              |188184 |
|MAZDA-OIL FILTERS         |184871 |
|GEAR OIL                  |139783 |
|HYUNDAI-AIR FILTERS       |136639 |
|TOYOTA-CABIN AIR FILTERS  |129464 |
|

+----------------------------------------------------+-------+
|product_sub_category                                |count  |
+----------------------------------------------------+-------+
|OIL FILTER                                          |8460754|
|PETROMIN SUPER 10W30 ENGINE OIL                     |4388761|
|PETROMIN SUPER SYNTHETIC 5W30 - 1L                  |2790319|
|PETROMIN SUPER BRAKE FLUID DOT-4                    |1350065|
|PETROMIN A-1 SUPER SYNTHETIC 5W-30 (6X4 LTR)        |1179304|
|AIR FILTER ELEMENT                                  |1101339|
|PETROMIN SUPER 10W30 1L                             |1048775|
|CABIN AIR FILTER                                    |900563 |
|ENGINE OIL FILTER                                   |830289 |
|PETROMIN FLEET MASTER LD 15W40 CH4                  |707336 |
|TOYOTA-OIL FILTERS                                  |641265 |
|PETROMIN ULTRA 7 20W50 ENGINE OIL                   |608135 |
|COOLANT GREEN                                       |6

In [22]:
base_sales.filter(
    f.col("product_code") == "NN165465RA0A"
).select(
    "product_code",
    "product_family",
    "product_category",
    "product_sub_category",
    "product_id",
).distinct().show(50, truncate=False)

+------------+-------------------------+------------------+--------------------+----------------------------------------------------------------------------------+
|product_code|product_family           |product_category  |product_sub_category|product_id                                                                        |
+------------+-------------------------+------------------+--------------------+----------------------------------------------------------------------------------+
|NN165465RA0A|VIP SERVICE CHARGES      |NISSAN-AIR FILTERS|AIR FILTER ELEMENT  |VIP SERVICE CHARGES | NISSAN-AIR FILTERS | AIR FILTER ELEMENT | NN165465RA0A      |
|NN165465RA0A|AIR FILTER SERVICES      |NISSAN-AIR FILTERS|AIR FILTER ELEMENT  |AIR FILTER SERVICES | NISSAN-AIR FILTERS | AIR FILTER ELEMENT | NN165465RA0A      |
|NN165465RA0A|CABIN AIR FILTER SERVICES|NISSAN-AIR FILTERS|AIR FILTER ELEMENT  |CABIN AIR FILTER SERVICES | NISSAN-AIR FILTERS | AIR FILTER ELEMENT | NN165465RA0A|
|NN165465RA0A|EX

In [ ]:
base_sales = base_sales.withColumn(
    "discount_percent",
    f.col("discount_amount") / (f.col("sales_amount_catalog"))
)

In [12]:
ftr_trx_sales = base_sales.groupBy(
    ["_id", "_observ_end_dt"]
).agg(
    f.sum("sales_amount").alias("month_total_sales"),
    f.sum("sales_amount_net").alias("month_net_sales"),
    f.sum("total_profit").alias("month_total_profit"),
    f.sum("net_cost").alias("month_total_cost"),
    f.sum("discount_amount").alias("month_total_amount_discounts"),
    f.sum("has_discount").alias("month_total_qty_discounts"),
    f.mean("discount_percent").alias("month_avg_percent_discount"),
    f.countDistinct("product_id").alias("month_distinct_skus_sold"),
    f.countDistinct("branch_id").alias("month_distinct_branches"),
    f.countDistinct("product_category").alias("month_distinct_product_categories"),
    f.countDistinct("transaction_id").alias("month_distinct_transactions"),
)

In [ ]:
# Compute average ticket, average discount and average sku's per order
ftr_trx_sales = ftr_trx_sales.withColumn(
    "month_total_discount_percentual",
    f.col("month_total_amount_discounts") / (f.col("month_net_sales") + f.col("month_total_amount_discounts"))
).withColumn(
    "month_avg_order",
    f.col("month_net_sales") / f.col("month_distinct_transactions")
).withColumn(
    "month_avg_skus_per_order",
    f.col("month_distinct_skus_sold") / f.col("month_distinct_transactions"),
).fillna(0)

In [14]:
ftr_trx_sales.filter(
    f.col("month_total_discount_percentual") > 1
).show(5, truncate=False)

+---------------------+--------------+------------------+------------------+------------------+------------------+----------------------------+-------------------------+--------------------------+------------------------+-----------------------+---------------------------------+---------------------------+-------------------------------+------------------+------------------------+
|_id                  |_observ_end_dt|month_total_sales |month_net_sales   |month_total_profit|month_total_cost  |month_total_amount_discounts|month_total_qty_discounts|month_avg_percent_discount|month_distinct_skus_sold|month_distinct_branches|month_distinct_product_categories|month_distinct_transactions|month_total_discount_percentual|month_avg_order   |month_avg_skus_per_order|
+---------------------+--------------+------------------+------------------+------------------+------------------+----------------------------+-------------------------+--------------------------+------------------------+-----------

In [15]:
ftr_trx_sales.filter(
    f.col("month_avg_percent_discount") > 1
).show(5, truncate=False)

+---------------------+--------------+------------------+------------------+------------------+------------------+----------------------------+-------------------------+--------------------------+------------------------+-----------------------+---------------------------------+---------------------------+-------------------------------+------------------+------------------------+
|_id                  |_observ_end_dt|month_total_sales |month_net_sales   |month_total_profit|month_total_cost  |month_total_amount_discounts|month_total_qty_discounts|month_avg_percent_discount|month_distinct_skus_sold|month_distinct_branches|month_distinct_product_categories|month_distinct_transactions|month_total_discount_percentual|month_avg_order   |month_avg_skus_per_order|
+---------------------+--------------+------------------+------------------+------------------+------------------+----------------------------+-------------------------+--------------------------+------------------------+-----------

# Last transaction features

In [118]:
ftr_last_trx_sales = base_sales.groupBy(
    ["_id", "_observ_end_dt", "transaction_id", "transaction_dt"]
).agg(
    f.sum("sales_amount").alias("last_trx_total_sales"),
    f.sum("sales_amount_net").alias("last_trx_net_sales"),
    f.sum("total_profit").alias("last_trx_total_profit"),
    f.sum("net_cost").alias("last_trx_total_cost"),
    f.sum("discount_amount").alias("last_trx_total_amount_discounts"),
    f.sum("has_discount").alias("last_trx_total_qty_discounts"),
    f.avg("current_mileage").alias("last_trx_current_mileage"),
    f.countDistinct("product_id").alias("last_trx_distinct_skus_sold"),
    f.countDistinct("branch_id").alias("last_trx_distinct_branches"),
    f.countDistinct("product_category").alias("last_trx_distinct_product_categories"),
    f.countDistinct("transaction_id").alias("last_trx_distinct_transactions"),
    # TODO: test on notebook
    # f.max(f.when(f.countDistinct("product_category") > 1, 1).otherwise(0)).alias("is_bundle"),
    f.max(f.when(f.lower("product_category").isin("oil", "oil synthetic"), 1).otherwise(0)).alias("has_oil"),
    f.max(f.when(f.contains(f.lower("product_category"), f.lit("oil filter")), 1).otherwise(0)).alias("has_oil_filter"),
    f.max(f.when(f.contains(f.lower("product_category"), f.lit("air filter")), 1).otherwise(0)).alias("has_air_filter"),
    f.max(f.when(f.contains(f.lower("product_category"), f.lit("ac ")), 1).otherwise(0)).alias("has_ac"),
    f.max(f.when(f.contains(f.lower("product_category"), f.lit("tyres")), 1).otherwise(0)).alias("has_tires"),
    f.max(f.when(f.contains(f.lower("product_category"), f.lit("batt")), 1).otherwise(0)).alias("has_batteries"),
    f.max(f.when(f.contains(f.lower("product_category"), f.lit("transmission")), 1).otherwise(0)).alias("has_transmission"),
    f.max(f.when(f.contains(f.lower("product_category"), f.lit("engine")), 1).otherwise(0)).alias("has_engine"),
    f.max(f.when(f.contains(f.lower("product_category"), f.lit("additives")), 1).otherwise(0)).alias("has_additives"),
    f.max(f.when(f.contains(f.lower("product_category"), f.lit("plug")), 1).otherwise(0)).alias("has_plug"),
    f.max(f.when(f.contains(f.lower("product_category"), f.lit("fuel")), 1).otherwise(0)).alias("has_fuel"),
    f.max(f.when(f.contains(f.lower("product_category"), f.lit("gear")), 1).otherwise(0)).alias("has_gear"),
).withColumn(
    "last_transaction_n_skus",
    f.col("last_trx_distinct_skus_sold") / f.col("last_trx_distinct_transactions"),
).fillna(0)

In [119]:
ftr_last_trx_sales.show(5)

+--------------------+--------------+--------------------+-------------------+--------------------+------------------+---------------------+-------------------+-------------------------------+----------------------------+------------------------+---------------------------+--------------------------+------------------------------------+------------------------------+-------+--------------+--------------+------+---------+-------------+----------------+----------+-------------+--------+--------+--------+-----------------------+
|                 _id|_observ_end_dt|      transaction_id|     transaction_dt|last_trx_total_sales|last_trx_net_sales|last_trx_total_profit|last_trx_total_cost|last_trx_total_amount_discounts|last_trx_total_qty_discounts|last_trx_current_mileage|last_trx_distinct_skus_sold|last_trx_distinct_branches|last_trx_distinct_product_categories|last_trx_distinct_transactions|has_oil|has_oil_filter|has_air_filter|has_ac|has_tires|has_batteries|has_transmission|has_engine|ha

In [149]:
w1 = Window.partitionBy("_id", "_observ_end_dt").orderBy(f.col("transaction_dt"))

ftr_last_trx_sales_filtered = ftr_last_trx_sales.withColumn(
    "last_transaction_id",
    f.last("transaction_id").over(w1.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))
).withColumn(
    "rn",
    f.row_number().over(w1)
).withColumn(
    "max_rn",
    f.max("rn").over(w1.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))
).filter(
    f.col("transaction_id") == f.col("last_transaction_id")
)

In [150]:
ftr_last_trx_sales_filtered.count()

11167679

In [154]:
ftr_last_trx_sales_filtered.groupBy(
    "_id", "_observ_end_dt"
).count().orderBy(f.desc("count"), "_id", "_observ_end_dt").show(20, truncate=False)

+--------------------------------------------------------------------------+--------------+-----+
|_id                                                                       |_observ_end_dt|count|
+--------------------------------------------------------------------------+--------------+-----+
|00000540-7986-4B85-A946-E1A9D913344F__F40BF538-A9A6-49E3-9623-9301AE9FD953|2023-04-01    |1    |
|00000847-CECD-43FB-A2F9-4D4B44A8BFFC__FF645321-C6D1-4864-91B7-F00EA37C30DB|2020-10-01    |1    |
|00000847-CECD-43FB-A2F9-4D4B44A8BFFC__FF645321-C6D1-4864-91B7-F00EA37C30DB|2021-01-01    |1    |
|00000847-CECD-43FB-A2F9-4D4B44A8BFFC__FF645321-C6D1-4864-91B7-F00EA37C30DB|2021-04-01    |1    |
|00000847-CECD-43FB-A2F9-4D4B44A8BFFC__FF645321-C6D1-4864-91B7-F00EA37C30DB|2021-07-01    |1    |
|00000847-CECD-43FB-A2F9-4D4B44A8BFFC__FF645321-C6D1-4864-91B7-F00EA37C30DB|2022-07-01    |1    |
|00000E2B-FD56-4F34-A54C-159E564F5F79__FF8A2477-4270-4547-B4CB-5706EA4479A0|2023-01-01    |1    |
|00000E2B-FD56-4F34-

In [152]:
ftr_last_trx_sales_filtered.filter(
    f.col("_id") == "000F803A-79A2-4666-B548-257B376B20C1__F45146CF-52D5-49C5-8BE1-9F3BE0C3AAD0"
).select(
    "_id", "_observ_end_dt", "transaction_id", "transaction_dt", "last_transaction_id", "rn", "max_rn"
).orderBy("_id", "_observ_end_dt").show(50, truncate=False)

+--------------------------------------------------------------------------+--------------+------------------------------------+-------------------+------------------------------------+---+------+
|_id                                                                       |_observ_end_dt|transaction_id                      |transaction_dt     |last_transaction_id                 |rn |max_rn|
+--------------------------------------------------------------------------+--------------+------------------------------------+-------------------+------------------------------------+---+------+
|000F803A-79A2-4666-B548-257B376B20C1__F45146CF-52D5-49C5-8BE1-9F3BE0C3AAD0|2022-01-01    |5BCB4221-F375-4742-B8CF-5944D80CDAB5|2022-03-28 19:38:30|5BCB4221-F375-4742-B8CF-5944D80CDAB5|2  |2     |
|000F803A-79A2-4666-B548-257B376B20C1__F45146CF-52D5-49C5-8BE1-9F3BE0C3AAD0|2022-04-01    |AAB18632-7751-46B0-AEAC-DB38F4FF4ADA|2022-06-08 11:29:34|AAB18632-7751-46B0-AEAC-DB38F4FF4ADA|2  |2     |
|000F803A-79A2-

In [156]:
ftr_last_trx_sales_filtered.count()

11167679

In [155]:
ftr_last_trx_sales_filtered.dropDuplicates(subset=["_id", "_observ_end_dt"]).count()

11167679

In [168]:
out = base_sales.select(
    "_id", "_observ_end_dt"
).distinct().join(
    ftr_last_trx_sales_filtered,
    on=["_id", "_observ_end_dt"],
    how="left"
).fillna(0).orderBy("_id", "_observ_end_dt")

In [ ]:
w_churn = Window.partitionBy("_id").orderBy(f.col("_observ_end_dt")).rowsBetween(1, 1)


In [ ]:
churn = out.withColumn(
    "is_active",
    f.when(f.col("last_trx_total_sales") > 0, 1).otherwise(0)
).withColumn(
    "target_churn",
    f.when(f.sum("is_active").over(w_churn) < 1, 1).otherwise(0)
)

In [178]:
churn.orderBy("_id", "_observ_end_dt").show(40, truncate=False)

+--------------------------------------------------------------------------+--------------+------------------------------------+-------------------+--------------------+------------------+---------------------+-------------------+-------------------------------+----------------------------+------------------------+---------------------------+--------------------------+------------------------------------+------------------------------+-------+--------------+--------------+------+---------+-------------+----------------+----------+-------------+--------+--------+--------+-----------------------+------------------------------------+---+------+---------+------------+--------------+--------------+
|_id                                                                       |_observ_end_dt|transaction_id                      |transaction_dt     |last_trx_total_sales|last_trx_net_sales|last_trx_total_profit|last_trx_total_cost|last_trx_total_amount_discounts|last_trx_total_qty_discounts|last_t

In [126]:
ftr_last_trx_sales_filtered.filter(
    f.col("_id") == "000119E4-727C-4518-8BB0-EBC6C167E11B__88BCDD68-9E41-40E6-93FE-4F33E392504D"
).select(
    "_id", "_observ_end_dt", "transaction_id", "transaction_dt", "last_transaction_id"
).orderBy("_id", "_observ_end_dt").show(50, truncate=False)

+--------------------------------------------------------------------------+--------------+------------------------------------+-------------------+------------------------------------+
|_id                                                                       |_observ_end_dt|transaction_id                      |transaction_dt     |last_transaction_id                 |
+--------------------------------------------------------------------------+--------------+------------------------------------+-------------------+------------------------------------+
|000119E4-727C-4518-8BB0-EBC6C167E11B__88BCDD68-9E41-40E6-93FE-4F33E392504D|2024-01-01    |0598DCC1-CF76-4D1C-9E02-59FFC24BC7C0|2024-01-05 13:33:11|0598DCC1-CF76-4D1C-9E02-59FFC24BC7C0|
|000119E4-727C-4518-8BB0-EBC6C167E11B__88BCDD68-9E41-40E6-93FE-4F33E392504D|2024-04-01    |0E9B31F5-1B39-4395-82E1-490D599EE28E|2024-06-18 16:14:30|0E9B31F5-1B39-4395-82E1-490D599EE28E|
|000119E4-727C-4518-8BB0-EBC6C167E11B__88BCDD68-9E41-40E6-93FE-4F33E39

In [81]:
transactions_data.filter(f.col("transaction_id") == "0E9B31F5-1B39-4395-82E1-490D599EE28E").show()

+--------------------+--------------------+--------------------+--------------------+-------------------+------------------+----------------+--------------------+--------------------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+------------+------------+---------------------+------------+-------------+------------------+--------------------+--------------+
|      transaction_id|         customer_id| customer_vehicle_id|           branch_id|     transaction_dt|    product_family|product_category|product_sub_category|          product_id|is_fleet|is_pms|current_mileage|previous_mileage|new_customer|new_vehicle|warranty|quantity|net_cost|discount_amount|sales_amount_net|sales_amount|total_profit|sales_amount_net_perc|has_discount|delta_mileage|delta_mileage_perc|                 _id|_observ_end_dt|
+--------------------+--------------------+--------------------+--------------------+-------------------

# Time features

In [33]:
ftr_trx_sales_time = base_sales.filter(
    f.col("transaction_id").isNotNull()
).dropDuplicates(subset=["_id", "_observ_end_dt", "transaction_id", "transaction_dt"]).withColumn(
    "dayofweek",
    f.dayofweek("transaction_dt")
).withColumn(
    "dayofmonth",
    f.dayofmonth("transaction_dt")
).withColumn(
    "is_weekday",
    f.when(f.col("dayofweek") < 6, 1).otherwise(0)
).withColumn(
    "is_weekend",
    f.when(f.col("dayofweek") > 5, 1).otherwise(0)
).withColumn(
    "week_of_month",
    f.date_format("transaction_dt", "W")
).withColumn(
    "hourofday",
    f.hour("transaction_dt")
).withColumn(
    "is_morning",
    f.when(
        (f.col("hourofday") > 5) & (f.col("hourofday") < 12),
        1
    ).otherwise(0)
).withColumn(
    "is_afternoon",
    f.when(
        (f.col("hourofday") > 11) & (f.col("hourofday") < 18),
        1
    ).otherwise(0)
).withColumn(
    "is_night",
    f.when(
        (f.col("hourofday") > 17) & (f.col("hourofday") < 24),
        1
    ).otherwise(0)
).withColumn(
    "is_afternight",
    f.when(
        (f.col("hourofday") > -1) & (f.col("hourofday") < 5),
        1
    ).otherwise(0)
).select(
    "_id",
    "_observ_end_dt",
    "transaction_id",
    "transaction_dt",
    "dayofweek",
    "dayofmonth",
    "is_weekday",
    "is_weekend",
    "week_of_month",
    "hourofday",
    "is_morning",
    "is_afternoon",
    "is_night",
    "is_afternight"
)

In [34]:
ftr_trx_sales_time.filter(f.col("transaction_id").isNotNull()).show(5)

+--------------------+--------------+--------------------+-------------------+---------+----------+----------+----------+-------------+---------+----------+------------+--------+-------------+
|                 _id|_observ_end_dt|      transaction_id|     transaction_dt|dayofweek|dayofmonth|is_weekday|is_weekend|week_of_month|hourofday|is_morning|is_afternoon|is_night|is_afternight|
+--------------------+--------------+--------------------+-------------------+---------+----------+----------+----------+-------------+---------+----------+------------+--------+-------------+
|000119E4-727C-451...|    2024-04-01|0E9B31F5-1B39-439...|2024-06-18 16:14:30|        3|        18|         1|         0|            4|       16|         0|           1|       0|            0|
|00013803-8018-44F...|    2022-07-01|1C715156-8AFF-43D...|2022-09-06 09:45:21|        3|         6|         1|         0|            2|        9|         1|           0|       0|            0|
|00013803-8018-44F...|    2022-07-0

In [35]:
ftr_trx_sales_time_week = ftr_trx_sales_time.groupBy(
    ["_id", "_observ_end_dt", "transaction_id"]
).pivot(
    "week_of_month",
).agg(
    f.countDistinct("transaction_id").alias("trx_distinct_transactions"),
).fillna(0)

columns = ["_id", "_observ_end_dt", "transaction_id"]

for col in ["1", "2", "3", "4", "5", "6"]:
    if col not in ["_id", "_observ_end_dt", "transaction_id"]:
        new_name = f"trx_distinct_transactions_week_{col}"

        ftr_trx_sales_time_week = ftr_trx_sales_time_week.withColumnRenamed(
            col,
            new_name,
        )

        if "null" not in col:
            columns.append(new_name)

ftr_trx_sales_time_week = ftr_trx_sales_time_week.select(columns)

In [36]:
ftr_trx_sales_time_week.withColumn(
    "rowSum",
    f.col("trx_distinct_transactions_week_1") +
    f.col("trx_distinct_transactions_week_2") +
    f.col("trx_distinct_transactions_week_3") +
    f.col("trx_distinct_transactions_week_4") +
    f.col("trx_distinct_transactions_week_5") +
    f.col("trx_distinct_transactions_week_6")
).filter(
    f.col("rowSum") > 1
).show(5)

+---+--------------+--------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+------+
|_id|_observ_end_dt|transaction_id|trx_distinct_transactions_week_1|trx_distinct_transactions_week_2|trx_distinct_transactions_week_3|trx_distinct_transactions_week_4|trx_distinct_transactions_week_5|trx_distinct_transactions_week_6|rowSum|
+---+--------------+--------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+------+
+---+--------------+--------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+------+



In [37]:
print("ftr_trx_sales_time", ftr_trx_sales_time.count())
print("ftr_trx_sales_time_week", ftr_trx_sales_time_week.count())

ftr_trx_sales_time 14197455


ftr_trx_sales_time_week 14197455


In [47]:
ftr_trx_sales_time = ftr_trx_sales_time.join(
    ftr_trx_sales_time_week,
    on=["_id", "_observ_end_dt", "transaction_id"],
    how="left"
)

In [48]:
ftr_trx_sales_time.count()

14197455

In [49]:
ftr_trx_sales_time.columns

['_id',
 '_observ_end_dt',
 'transaction_id',
 'transaction_dt',
 'dayofweek',
 'dayofmonth',
 'is_weekday',
 'is_weekend',
 'week_of_month',
 'hourofday',
 'is_morning',
 'is_afternoon',
 'is_night',
 'is_afternight',
 'trx_distinct_transactions_week_1',
 'trx_distinct_transactions_week_2',
 'trx_distinct_transactions_week_3',
 'trx_distinct_transactions_week_4',
 'trx_distinct_transactions_week_5',
 'trx_distinct_transactions_week_6']

In [51]:
ftr_trx_sales_time_agg = ftr_trx_sales_time.groupBy(
    ["_id", "_observ_end_dt"]
).agg(
    f.avg('dayofweek').alias("avg_day_of_week"),
    f.mode('dayofweek').alias("most_frequent_day_of_week"),
    f.avg('dayofmonth').alias("avg_day_of_month"),
    f.mode('dayofmonth').alias("most_frequent_day_of_month"),
    f.sum('is_weekday').alias("total_transactions_at_weekday"),
    f.sum('is_weekend').alias("total_transactions_at_weekend"),
    f.avg('week_of_month').alias("avg_week_transaction"),
    f.mode('week_of_month').alias("most_frequent_week_transaction"),
    f.avg('hourofday').alias("avg_hour_transaction"),
    f.mode('hourofday').alias("most_frequent_hour_transaction"),
    f.sum('is_morning').alias("total_transactions_at_morning"),
    f.sum('is_afternoon').alias("total_transactions_at_afternoon"),
    f.sum('is_night').alias("total_transactions_at_night"),
    f.sum('is_afternight').alias("total_transactions_at_afternight"),
    f.sum('trx_distinct_transactions_week_1').alias("total_transactions_week_1"),
    f.sum('trx_distinct_transactions_week_2').alias("total_transactions_week_2"),
    f.sum('trx_distinct_transactions_week_3').alias("total_transactions_week_3"),
    f.sum('trx_distinct_transactions_week_4').alias("total_transactions_week_4"),
    f.sum('trx_distinct_transactions_week_5').alias("total_transactions_week_5"),
    f.sum('trx_distinct_transactions_week_6').alias("total_transactions_week_6")
)

In [52]:
ftr_trx_sales_time.count()

14197455

In [55]:
ftr_trx_sales_time.select(f.countDistinct("_id"), f.countDistinct("_observ_end_dt")).show()

+-------------------+------------------------------+
|count(DISTINCT _id)|count(DISTINCT _observ_end_dt)|
+-------------------+------------------------------+
|            3254437|                            18|
+-------------------+------------------------------+



In [63]:
out = base_sales.select(
    "_id", "_observ_end_dt"
).distinct().join(
    ftr_trx_sales_time_agg,
    on=["_id", "_observ_end_dt"],
    how="left"
).fillna(0).orderBy("_id", "_observ_end_dt")

In [64]:
out.select(f.countDistinct("_id"), f.countDistinct("_observ_end_dt")).show()

+-------------------+------------------------------+
|count(DISTINCT _id)|count(DISTINCT _observ_end_dt)|
+-------------------+------------------------------+
|            3254437|                            18|
+-------------------+------------------------------+



In [65]:
print(prm_spine.count())
print(base_sales.count())
print(ftr_trx_sales_time_agg.count())
print(out.count())

37047207


67215134


11167679


37047207


In [66]:
out.dropDuplicates(subset=["_id", "_observ_end_dt"]).count()

37047207

# Vehicle features

In [184]:
from pyspark.sql.types import StringType

In [183]:
prm_customers = spark.read.parquet("../data/03_primary/prm_customers")
prm_vehicles = spark.read.parquet("../data/03_primary/prm_vehicles")

In [180]:
def categorizer(age):
    if age is None:
        return "None"
    elif age < 3:
        return "0_2"
    elif age < 6:
        return "3_5"
    elif age < 10:
        return "6_9"
    else:
        return "10_plus"

In [193]:
bucket_udf = f.udf(categorizer, StringType() )

prm_vehicles = prm_vehicles.withColumn(
    "_id",
    # f.col("customer_id")
    f.concat_ws("__", "customer_id", "customer_vehicle_id")
)

prm_customer_vehicles = prm_customers.join(
    prm_vehicles,
    on="customer_id",
    how="inner"
)

ftr_vehicles = base_sales.select(
    "_id", "_observ_end_dt", "transaction_id", "transaction_dt"
).distinct().join(
    prm_customer_vehicles,
    on=["_id",],
    how="left"
)

In [194]:
print(prm_spine.count())
print(prm_vehicles.count())
print(prm_customer_vehicles.count())
print(base_sales.count())
print(ftr_vehicles.count())

37047207
6977130


6785049


67215134


40076983


In [195]:
ftr_vehicles = ftr_vehicles.withColumn(
    "car_age",
    f.year(f.col("transaction_dt")).cast("int") - f.col("model_year").cast("int")
).withColumn(
    "car_age_group",
    bucket_udf(f.col("car_age"))
).withColumn(
    "car_group",
    f.concat(f.lower(f.col("vehicle_brand_level")), f.lit("__"), f.col("car_age_group"))
)

In [196]:
car_price_levels = [f"car_brand__{col}" for col in ["very_low", "low", "medium", "high", "very_high"]]
car_age_levels = [f"car_age__{col}" for col in [None, "0_2", "3_5", "6_9", "10_plus"]]
car_price_age_levels = [
    f"{price}__{age}"
    for price in car_price_levels
    for age in car_age_levels
]

In [197]:
ftr_vehicles_price_pivot = ftr_vehicles.dropna(
    subset=["_id", "transaction_id"]
).withColumn(
    "vehicle_brand_level",
    f.concat(f.lit("car_brand__"), f.lower(f.col("vehicle_brand_level")))
).groupBy(
    "_id", "transaction_id"
).pivot(
    "vehicle_brand_level", values=car_price_levels
).agg(
    f.countDistinct("customer_id").alias("count")
).fillna(0)

In [198]:
ftr_vehicles_age_pivot = ftr_vehicles.dropna(
    subset=["_id", "transaction_id"]
).withColumn(
    "car_age_group",
    f.concat(f.lit("car_age__"), f.col("car_age_group"))
).groupBy(
    "_id", "transaction_id"
).pivot(
    "car_age_group", values=car_age_levels
).agg(
    f.countDistinct("customer_id").alias("count")
).fillna(0)

In [199]:
ftr_vehicles_extended = ftr_vehicles.join(
    ftr_vehicles_price_pivot,
    on=["_id", "transaction_id"]
).join(
    ftr_vehicles_age_pivot,
    on=["_id", "transaction_id"]
)

In [200]:
print(prm_spine.count())
print(prm_vehicles.count())
print(prm_customer_vehicles.count())
print(base_sales.count())
print(ftr_vehicles.count())
print(ftr_vehicles_extended.count())

37047207
6977130


6785049


67215134


40076983


14197455


In [201]:
ftr_vehicles_selected = ftr_vehicles_extended.withColumn(
    "is_toyota",
    f.when(f.lower("maker") == "toyota", 1).otherwise(0)
).withColumn(
    "is_japanese",
    f.when(f.lower("maker").isin(["toyota", "nissan", "hyundai", "lexus", "honda", "mazda", "suzuki"]), 1).otherwise(0)
).withColumn(
    "is_american",
    f.when(f.lower("maker").isin(["ford", "chevrolet", "mercury"]), 1).otherwise(0)
).withColumn(
    "is_truck",
    f.when(f.contains(f.col("maker"), f.lit("trucks")), 1).otherwise(0)
).select(
    "_id",
    "_observ_end_dt",
    "car_age",
    "car_age_group",
    *car_price_levels,
    *car_age_levels,
    "is_toyota",
    "is_japanese",
    "is_american",
    "is_truck"
).orderBy("_id", "_observ_end_dt",)

In [202]:
print(prm_spine.count())
print(prm_vehicles.count())
print(prm_customer_vehicles.count())
print(base_sales.count())
print(ftr_vehicles.count())
print(ftr_vehicles_extended.count())
print(ftr_vehicles_selected.count())

37047207
6977130


6785049


67215134


40076983


14197455


14197455


In [215]:
out = base_sales.select(
    "_id", "_observ_end_dt"
).distinct().join(
    ftr_vehicles_selected.drop_duplicates(subset=["_id", "_observ_end_dt"]),
    on=["_id", "_observ_end_dt"],
    how="left"
)

In [216]:
out.count()

37047207

In [212]:
ftr_vehicles_selected.drop_duplicates(subset=["_id", "_observ_end_dt"]).count()

11167679

# Mileage model

In [ ]:
working_ids = [
    "403821AD-A894-4257-B8E1-63F1C14EFFBB__22EA1D18-6002-4F37-A2FF-CC088300DF12",
    "00000540-7986-4B85-A946-E1A9D913344F__F40BF538-A9A6-49E3-9623-9301AE9FD953",
    "00000E2B-FD56-4F34-A54C-159E564F5F79__FF8A2477-4270-4547-B4CB-5706EA4479A0",
    "00000E2B-FD56-4F34-A54C-159E564F5F79__FF8A2477-4270-4547-B4CB-5706EA4479A0",
    "00001CA5-294A-4B26-A4A1-A77D7B5B18CC__9946561B-097B-441A-BB89-95AF12D27759",
    "00001CA5-294A-4B26-A4A1-A77D7B5B18CC__9946561B-097B-441A-BB89-95AF12D27759",
]

working_ids = [
    "A4F50776-4238-465E-AE0D-1C8C376CAEE2__37D422A0-E176-4D4F-8721-3B231E1ABAA0",
    "A4F507F5-769F-4C5E-9D3A-CBDD668AB6C8__4124CB72-D524-4A5E-8806-2D741083C98A",
    "A4F508D3-F915-4881-92E1-64E425B7FA71__DE8981C4-D7CF-46AC-8E18-5A844E1B5F7F",
    "A4F510D7-F015-432A-ABCB-E72594B42014__BD2B7690-017A-4437-9F8C-85518A3CF131",
    "A4F51822-9DBE-4CEF-834F-F5BC2DC179EC__B089C48B-85D1-4E8C-BBFC-33BEA68D5027",
    "A4F52279-0F64-4165-BB89-600CD3314D59__4D5A9841-55D7-492F-B695-78DE31B6A760", ## GOOD ID
    "A4F553E8-D26B-4BB7-90E3-75D3F2950064__D71989F6-16FF-48D5-919B-4BA2286EF08D", ## GOOD ID
]

working_ids = [
    "ADF9EA7B-950A-4DF8-B3E4-FA44F535916A__E89BEB7A-E3C4-49B1-B137-5347EE53626D",
    "19D9BC46-B71F-46E1-8A7D-5A4F48AAE2FF__6B23EF10-4823-4457-A516-E3A69F2139BF",
    "7AE928E6-4908-4E9A-9B83-93B773ABC398__92F5C240-D131-43FF-B705-024580651AF3",
    "884884AE-CAAF-4257-B425-13E8FF898B6F__F0BA9438-43A9-457A-9E83-25D9443362CD"
]


# Negative values
working_ids = [
    "B3A8C9F7-653F-4EDD-A9BB-43A1F78954BA__885CCCBF-9231-4F44-BB48-9742A3F40609",
    "81FD6E6F-27CB-42A6-AAF0-882F8A255DDB__B650A67C-B8FC-4730-8A86-3D4A481BABA0",
    "1B155384-09D1-478C-9EAA-99AC2D328D29__4A99289D-A72E-48AE-86B2-AD8A5E57FFD0",
    "B7C56453-865E-4417-BB3C-1BEEB4FF5816__6B6B5CB2-08B9-4182-AAC7-13D8D14B7BCB",
    "A4629DC6-6624-4D63-99AC-5D620DAFE670__507B34B6-F94A-4D27-BF3F-1543EFD6135A",
    "50908F70-D5E8-42A4-98E0-95D9589FF6DE__71404FD4-5CCF-486B-AC6C-60EDCA23E8F0"
]

In [14]:
win_id = Window.partitionBy("_id")
win_id_monthly = Window.partitionBy("_id").orderBy("_observ_end_dt")

In [15]:
base_sales = base_sales.withColumn(
    "major_category",
    f.when(
        f.lower("product_category").isin("oil"),
        "oil"
    ).when(
        f.lower("product_category").isin("oil synthetic"),
        "oil_synthetic"
    ).when(
        f.contains(f.lower("product_category"), f.lit("oil filter")),
        "oil_filter"
    ).when(
        f.contains(f.lower("product_category"), f.lit("air filter")),
        "air_filter"
    ).when(
        f.contains(f.lower("product_category"), f.lit("ac ")),
        "ac"
    ).when(
        f.contains(f.lower("product_category"), f.lit("tyres")),
        "tyres"
    ).when(
        f.contains(f.lower("product_category"), f.lit("batt")),
        "batteries"
    ).when(
        f.contains(f.lower("product_category"), f.lit("transmission")),
        "transmission"
    ).when(
        f.contains(f.lower("product_category"), f.lit("engine")),
        "engine"
    ).when(
        f.contains(f.lower("product_category"), f.lit("additives")),
        "additives"
    ).when(
        f.contains(f.lower("product_category"), f.lit("plug")),
        "plug"
    ).when(
        f.contains(f.lower("product_category"), f.lit("fuel")),
        "fuel"
    ).when(
        f.contains(f.lower("product_category"), f.lit("gear")),
        "gear"
    ).when(
        f.contains(f.lower("product_category"), f.lit("brake")),
        "brake"
    ).when(
        f.contains(f.lower("product_category"), f.lit("coolant")),
        "coolant"
    ).otherwise("others")
)

In [16]:
major_categories_list = ["oil", "oil_synthetic", "oil_filter", "air_filter", "ac", "tyres", "batteries", "transmission", "engine", "additives", "plug", "fuel", "gear", "brake", "coolant"]
# major_categories_list = ["oil", "oil_filter", "transmission", "engine", "plug", "brake", "coolant"]
# major_categories_list = ["brake",]

In [17]:
category_sales = base_sales.filter(
    # f.col("_id") == working_ids[1]
    f.lit(True) == True
).filter(
    f.col("transaction_id").isNotNull()
    # f.lit(True) == True
).groupBy(
    "_id", "_observ_end_dt",
).pivot(
    "major_category"
).agg(
    f.last("transaction_dt", ignorenulls=True).alias("transaction_dt"), 
    f.last("current_mileage", ignorenulls=True).alias("current_mileage"),
)

category_sales.orderBy("_id", "_observ_end_dt").show(50, truncate=False)

25/05/09 20:31:14 WARN Column: Constructing trivially true equals predicate, 'true = true'. Perhaps you need to use aliases.
25/05/09 20:31:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------------------------------------------------------------+--------------+-------------------+------------------+------------------------+-------------------------+-------------------------+--------------------------+------------------------+-------------------------+--------------------+---------------------+----------------------+-----------------------+---------------------+----------------------+-------------------+--------------------+-------------------+--------------------+-------------------+-------------------+-------------------------+--------------------------+----------------------------+-----------------------------+---------------------+----------------------+-------------------+--------------------+---------------------------+----------------------------+--------------------+---------------------+
|_id                                                                       |_observ_end_dt|ac_transaction_dt  |ac_current_mileage|additives_transaction_dt|addi

In [18]:
# win_id_last_3 = Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(-3, -1)
# win_id.orderBy("transaction_dt").rowsBetween(-3, -1)
# win_id_next_3 = Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(1, 3)
# win_id.orderBy("transaction_dt").rowsBetween(1, 3)
# win_id.orderBy("transaction_dt").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
# win_id.orderBy("transaction_dt").rowsBetween(Window.unboundedPreceding, -1)

In [19]:

for cat in major_categories_list:
    category_sales = category_sales.orderBy("_id", "_observ_end_dt").withColumn(
    #     f"{cat}_test_info_i",
    #     f.abs(f.col(f"{cat}_current_mileage") - f.avg(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(1, 3)))
    # ).withColumn(
    #     f"{cat}_test_info_ii",
    #     f.std(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))
    # ).withColumn(
    #     f"{cat}_test_info_iii",
    #     f.avg(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(1, 3)).isNotNull()
    # ).withColumn(
    #     f"{cat}_test_info_res_i",
    #     (f.abs(f.col(f"{cat}_current_mileage") - f.avg(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(1, 3))) > f.std(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))), # & (f.avg(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(1, 3)).isNotNull()),
    # ).withColumn(
    #     f"{cat}_test_info_iv",
    #     (f.col(f"{cat}_current_mileage") - f.last(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))) / 
    #     f.when(f.date_diff(f.col(f"{cat}_transaction_dt"), f.last(f"{cat}_transaction_dt").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))) == 0, 1).otherwise(f.date_diff(f.col(f"{cat}_transaction_dt"), f.last(f"{cat}_transaction_dt").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))))
    # ).withColumn(
    #     f"{cat}_current_mileage",
    #     f.when(
    #         (f.abs(f.col(f"{cat}_current_mileage") - f.avg(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-3, -1))) > f.std(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))) & (f.avg(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-3, -1)).isNotNull()),
    #         None
    #     ).when(
    #         (f.abs(f.col(f"{cat}_current_mileage") - f.avg(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(1, 3))) > f.std(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))), # & (f.avg(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(1, 3)).isNotNull()),
    #         None
    #     ).otherwise(f.col(f"{cat}_current_mileage"))
    # ).withColumn(
        f"{cat}_current_mileage",
        f.when(
            (
                (f.col(f"{cat}_current_mileage") - f.last(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))) / 
                f.when(f.date_diff(f.col(f"{cat}_transaction_dt"), f.last(f"{cat}_transaction_dt").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))) == 0, 1).otherwise(f.date_diff(f.col(f"{cat}_transaction_dt"), f.last(f"{cat}_transaction_dt").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))))
            ) < 0,
            None
        ).otherwise(f.col(f"{cat}_current_mileage"))
    ).withColumn(
        f"{cat}_current_mileage",
        f.when(
            (
                (f.col(f"{cat}_current_mileage") - f.last(f"{cat}_current_mileage").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))) / 
                f.when(f.date_diff(f.col(f"{cat}_transaction_dt"), f.last(f"{cat}_transaction_dt").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))) == 0, 1).otherwise(f.date_diff(f.col(f"{cat}_transaction_dt"), f.last(f"{cat}_transaction_dt").over(win_id.orderBy(f"{cat}_transaction_dt").rowsBetween(-12, -1))))
            ) > 1200,
            None
        ).otherwise(f.col(f"{cat}_current_mileage"))
    ).withColumn(
        f"{cat}_transaction_dt",
        f.when(
            f.col(f"{cat}_current_mileage").isNull(), None
        ).otherwise(f.col(f"{cat}_transaction_dt"))
    ).withColumn(
        f"{cat}_last_current_mileage",
        f.last(f"{cat}_current_mileage", ignorenulls=True).over(win_id_monthly.rowsBetween(-12, -1))
    ).withColumn(
        f"{cat}_last_transaction_dt",
        f.last(f"{cat}_transaction_dt", ignorenulls=True).over(win_id_monthly.rowsBetween(-12, -1))
    ).withColumn(
        f"{cat}_run_mileage",
        f.col(f"{cat}_current_mileage") - f.col(f"{cat}_last_current_mileage")
    ).withColumn(
        f"{cat}_run_period",
        f.date_diff(f"{cat}_transaction_dt", f"{cat}_last_transaction_dt")
    ).withColumn(
        f"{cat}_mileage_per_day",
        f.col(f"{cat}_run_mileage") / f.col(f"{cat}_run_period")
    )

category_sales.orderBy("_id", "_observ_end_dt").show(50, truncate=False)

+--------------------------------------------------------------------------+--------------+-------------------+------------------+------------------------+-------------------------+-------------------------+--------------------------+------------------------+-------------------------+--------------------+---------------------+----------------------+-----------------------+---------------------+----------------------+-------------------+--------------------+-------------------+--------------------+-------------------+-------------------+-------------------------+--------------------------+----------------------------+-----------------------------+---------------------+----------------------+-------------------+--------------------+---------------------------+----------------------------+--------------------+---------------------+------------------------+-----------------------+---------------+--------------+-------------------+----------------------------------+-------------------------

In [20]:
agg_cols = []
for cat in major_categories_list:
    cat_list = [
        f.last(f"{cat}_transaction_dt", ignorenulls=True).alias(f"{cat}_last_transaction_dt"),
        f.min(f"{cat}_current_mileage").alias(f"{cat}_month_min_current_mileage"),
        f.max(f"{cat}_current_mileage").alias(f"{cat}_month_max_current_mileage"),
        f.mean(f"{cat}_run_mileage").alias(f"{cat}_month_avg_run_mileage"),
        f.mean(f"{cat}_run_period").alias(f"{cat}_month_avg_run_period"),
        f.mean(f"{cat}_mileage_per_day").alias(f"{cat}_month_avg_mileage_per_day"),
    ]
    agg_cols += cat_list

grouped_category_sales = category_sales.groupBy(
"_id", "_observ_end_dt",
).agg(
    *agg_cols
)

for cat in major_categories_list:
    grouped_category_sales = grouped_category_sales.withColumn(
        f"{cat}_month_avg_mileage_per_day",
        f.when(
            f.col(f"{cat}_month_avg_mileage_per_day").isNull(),
            f.round(f.first(f"{cat}_month_avg_mileage_per_day", ignorenulls=True).over(win_id_monthly.rowsBetween(0, Window.unboundedFollowing)), 2)
        ).otherwise(f.col(f"{cat}_month_avg_mileage_per_day"))
    )

grouped_category_sales.show(50, truncate=False)

+--------------------------------------------------------------------------+--------------+-----------------------+-----------------------------+-----------------------------+-------------------------+------------------------+-----------------------------+---------------------------------+---------------------------------------+---------------------------------------+-----------------------------------+----------------------------------+---------------------------------------+------------------------------+------------------------------------+------------------------------------+--------------------------------+-------------------------------+------------------------------------+------------------------------+------------------------------------+------------------------------------+--------------------------------+-------------------------------+------------------------------------+----------------------+----------------------------+----------------------------+------------------------

In [27]:
base_sales.select(
    "_id", "_observ_end_dt"
).distinct().join(
    grouped_category_sales,
    on=["_id", "_observ_end_dt"],
    how="left"
).filter(
    f.col("_observ_end_dt") == "2024-12-31"
).select(
    # f.sum(
    #     f.when(
    #         f.col("oil_month_avg_run_mileage").isNull(),
    #         1
    #     ).otherwise(0)
    # ).alias("null_clients"),
    # f.sum(
    #     f.when(
    #         f.col("oil_month_avg_run_mileage").isNotNull(),
    #         1
    #     ).otherwise(0)
    # ).alias("active_clients"),
    f.min(f.col("oil_month_avg_run_mileage")).alias("overall_min"),
    f.mean(f.col("oil_month_avg_run_mileage")).alias("overall_mean"),
    f.max(f.col("oil_month_avg_run_mileage")).alias("overall_max"),
).show()

+-----------+------------------+-----------+
|overall_min|      overall_mean|overall_max|
+-----------+------------------+-----------+
| -1691076.0|10522.963488411724|  9000000.0|
+-----------+------------------+-----------+



In [28]:
grouped_category_sales.filter(
    f.col("_observ_end_dt") == "2024-12-31"
).filter(
    f.col("oil_month_avg_run_mileage") < 0
).show(20, truncate=False)

+--------------------------------------------------------------------------+--------------+-----------------------+-----------------------------+-----------------------------+-------------------------+------------------------+-----------------------------+---------------------------------+---------------------------------------+---------------------------------------+-----------------------------------+----------------------------------+---------------------------------------+------------------------------+------------------------------------+------------------------------------+--------------------------------+-------------------------------+------------------------------------+------------------------------+------------------------------------+------------------------------------+--------------------------------+-------------------------------+------------------------------------+----------------------+----------------------------+----------------------------+------------------------

In [31]:
working_ids = [
    "B3A8C9F7-653F-4EDD-A9BB-43A1F78954BA__885CCCBF-9231-4F44-BB48-9742A3F40609",
    "81FD6E6F-27CB-42A6-AAF0-882F8A255DDB__B650A67C-B8FC-4730-8A86-3D4A481BABA0",
    "1B155384-09D1-478C-9EAA-99AC2D328D29__4A99289D-A72E-48AE-86B2-AD8A5E57FFD0",
    "B7C56453-865E-4417-BB3C-1BEEB4FF5816__6B6B5CB2-08B9-4182-AAC7-13D8D14B7BCB",
    "A4629DC6-6624-4D63-99AC-5D620DAFE670__507B34B6-F94A-4D27-BF3F-1543EFD6135A",
    "50908F70-D5E8-42A4-98E0-95D9589FF6DE__71404FD4-5CCF-486B-AC6C-60EDCA23E8F0"
]

In [108]:
mileage_sales = base_sales.groupBy(
    "_id", "_observ_end_dt", "transaction_id", "transaction_dt"
).agg(
    f.mean("current_mileage").alias("current_mileage"),
    f.max("current_mileage").alias("max_mileage")
).filter(
    f.col("transaction_id").isNotNull()
)

In [109]:
win_id_all = Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
win_id_last_3 = Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(-3, -1)
win_id_next_3 = Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(1, 3)

In [117]:
x = mileage_sales.withColumn(
    "current_mileage",
    f.when(
        (f.col("current_mileage").isNotNull()) &
        (f.max("current_mileage").over(Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(Window.unboundedPreceding, -1)) > 900000) &
        (f.col("current_mileage") < 900000),
        f.col("current_mileage") + 1000000
    ).otherwise(f.col("current_mileage"))
).withColumn(
    "current_mileage",
    f.when(
        (f.abs(f.col("current_mileage") - f.avg("current_mileage").over(win_id_last_3)) > f.std("current_mileage").over(win_id_all)) & (f.avg("current_mileage").over(win_id_last_3).isNotNull()),
        None
    ).when(
        (f.abs(f.col("current_mileage") - f.avg("current_mileage").over(win_id_next_3)) > f.std("current_mileage").over(win_id_all)) & (f.avg("current_mileage").over(win_id_next_3).isNotNull()),
        None
    ).otherwise(f.col("current_mileage"))
).withColumn(
    "current_mileage",
    f.when(
        (
            (f.col("current_mileage") - f.last("current_mileage").over(win_id)) / 
            f.when(f.date_diff(f.col("transaction_dt"), f.last("transaction_dt").over(win_id)) == 0, 1).otherwise(f.date_diff(f.col("transaction_dt"), f.last("transaction_dt").over(win_id)))
        ) > 300,
        None
    ).otherwise(f.col("current_mileage"))
).withColumn(
    "current_mileage",
    f.when(
        (
            (f.col("current_mileage") - f.last("current_mileage").over(win_id)) / 
            f.when(f.date_diff(f.col("transaction_dt"), f.last("transaction_dt").over(win_id)) == 0, 1).otherwise(f.date_diff(f.col("transaction_dt"), f.last("transaction_dt").over(win_id)))
        ) < 0,
        None
    ).otherwise(f.col("current_mileage"))
).withColumn(
    "transaction_dt",
    f.when(
        f.col("current_mileage").isNull(),
        None
    ).otherwise(f.col("transaction_dt"))
).withColumn(
    "last_transaction_dt",
    f.last("transaction_dt", ignorenulls=True).over(Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(Window.unboundedPreceding, -1)),
).withColumn(
    "last_current_mileage",
    f.last("current_mileage", ignorenulls=True).over(Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(Window.unboundedPreceding, -1)),
).withColumn(
    "run_mileage",
    f.col("current_mileage") - f.col("last_current_mileage")
).withColumn(
    "run_period",
    f.date_diff("transaction_dt", "last_transaction_dt")
).withColumn(
    "mpd",
    f.col("run_mileage") / f.col("run_period")
).withColumn(
    "mpd",
    f.when(
        f.col("mpd").isNull(),
        f.round(f.first("mpd", ignorenulls=True).over(Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(0, Window.unboundedFollowing)))
    ).otherwise(f.col("mpd"))
).withColumn(
    "current_mileage",
    f.when(
        f.col("mpd") > 300,
        None
    ).otherwise(f.col("current_mileage"))
).withColumn(
    "transaction_dt",
    f.when(
        f.col("current_mileage").isNull(),
        None
    ).otherwise(f.col("transaction_dt"))
).withColumn(
    "last_transaction_dt",
    f.last("transaction_dt", ignorenulls=True).over(Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(Window.unboundedPreceding, -1)),
).withColumn(
    "last_current_mileage",
    f.last("current_mileage", ignorenulls=True).over(Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(Window.unboundedPreceding, -1)),
).withColumn(
    "run_mileage",
    f.col("current_mileage") - f.col("last_current_mileage")
).withColumn(
    "run_period",
    f.date_diff("transaction_dt", "last_transaction_dt")
).withColumn(
    "mpd",
    f.col("run_mileage") / f.col("run_period")
).withColumn(
    "mpd",
    f.when(
        f.col("mpd").isNull(),
        f.round(f.first("mpd", ignorenulls=True).over(Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(0, Window.unboundedFollowing)))
    ).otherwise(f.col("mpd"))
).withColumn(
    "current_mileage",
    f.when(
        f.col("mpd") < 0,
        None
    ).otherwise(f.col("current_mileage"))
).withColumn(
    "transaction_dt",
    f.when(
        f.col("current_mileage").isNull(),
        None
    ).otherwise(f.col("transaction_dt"))
).withColumn(
    "last_transaction_dt",
    f.last("transaction_dt", ignorenulls=True).over(Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(Window.unboundedPreceding, -1)),
).withColumn(
    "last_current_mileage",
    f.last("current_mileage", ignorenulls=True).over(Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(Window.unboundedPreceding, -1)),
).withColumn(
    "run_mileage",
    f.col("current_mileage") - f.col("last_current_mileage")
).withColumn(
    "run_period",
    f.date_diff("transaction_dt", "last_transaction_dt")
).withColumn(
    "mpd",
    f.col("run_mileage") / f.col("run_period")
).withColumn(
    "mpd",
    f.when(
        f.col("mpd").isNull(),
        f.round(f.first("mpd", ignorenulls=True).over(Window.partitionBy("_id").orderBy("transaction_dt").rowsBetween(0, Window.unboundedFollowing)))
    ).otherwise(f.col("mpd"))
)

x.select(
    f.mean(
        f.when(
            (f.col("mpd") > 300),
            1
        ).otherwise(0)
    ),
    f.mean(
        f.when(
            f.col("mpd") < 0,
            1
        ).otherwise(0)
    ),
    f.round(f.min("mpd")),
    f.round(f.median("mpd")),
    f.round(f.mean("mpd")),
    f.round(f.max("mpd")),
).show()

+--------------------------------------------+------------------------------------------+------------------+---------------------+------------------+------------------+
|avg(CASE WHEN (mpd > 300) THEN 1 ELSE 0 END)|avg(CASE WHEN (mpd < 0) THEN 1 ELSE 0 END)|round(min(mpd), 0)|round(median(mpd), 0)|round(avg(mpd), 0)|round(max(mpd), 0)|
+--------------------------------------------+------------------------------------------+------------------+---------------------+------------------+------------------+
|                        0.004254370056153179|                      2.462847278071868E-4|         -431621.0|                 97.0|             109.0|          655629.0|
+--------------------------------------------+------------------------------------------+------------------+---------------------+------------------+------------------+



In [121]:
working_ids = [
    "00085C8C-BC2A-4DBA-B083-C56A47C27525__8DA35057-0961-4744-A97F-499F45E47BF0",
    "000BECC8-C481-4836-AFB4-A546BE802A4F__0C4F8AE6-E0FA-4DDA-A809-85A6032CEA8B",
    "000F803A-79A2-4666-B548-257B376B20C1__F45146CF-52D5-49C5-8BE1-9F3BE0C3AAD0",
]

In [125]:
x.filter(
    f.col("_id") == working_ids[2]
).orderBy(
    "_id", "_observ_end_dt", "transaction_dt"
).show(50, truncate=False)

+--------------------------------------------------------------------------+--------------+------------------------------------+-------------------+---------------+-----------+-------------------+--------------------+-----------+----------+------------------+
|_id                                                                       |_observ_end_dt|transaction_id                      |transaction_dt     |current_mileage|max_mileage|last_transaction_dt|last_current_mileage|run_mileage|run_period|mpd               |
+--------------------------------------------------------------------------+--------------+------------------------------------+-------------------+---------------+-----------+-------------------+--------------------+-----------+----------+------------------+
|000F803A-79A2-4666-B548-257B376B20C1__F45146CF-52D5-49C5-8BE1-9F3BE0C3AAD0|2022-03-31    |B3441512-920B-4610-8064-DF4DD327C653|2022-03-05 00:17:29|341552.0       |341552.0   |NULL               |NULL                |NUL

25/05/10 11:29:11 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 997546 ms exceeds timeout 120000 ms
25/05/10 11:29:11 WARN SparkContext: Killing executors is not supported by current scheduler.
25/05/10 11:46:55 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o

In [ ]:
tp = 171051.4
tn = 211968.9
fp = 29229.6
fn = 63305.6
total = tp + tn + fp + fn
print("tp:", tp/total)
print("tn:", tn/total)
print("fp:", fp/total)
print("fn:", fn/total)


tp: 0.3596875653840614
tn: 0.44572904739825325
fp: 0.061464119329920484
fn: 0.13311926788776496


In [ ]:
base_sales.filter(
    f.col("_id") == working_ids[1]
).select(
    "_id", "_observ_end_dt", "transaction_id", "transaction_dt", "current_mileage", "major_category"
).show(50, truncate=False)

In [21]:
# ftr_trx_mileage_product = 
base_sales.filter(
    f.col("transaction_id").isNotNull()
).select(
    "_id", "_observ_end_dt", "transaction_id", "transaction_dt", "current_mileage", "product_category"
).show(5, truncate=False)

# .groupBy(
#     "_id", "_observ_end_dt", "transaction_id", "transaction_dt"
# )

+--------------------------------------------------------------------------+--------------+------------------------------------+-------------------+---------------+-----------------+
|_id                                                                       |_observ_end_dt|transaction_id                      |transaction_dt     |current_mileage|product_category |
+--------------------------------------------------------------------------+--------------+------------------------------------+-------------------+---------------+-----------------+
|00003D26-F6BB-457D-9CCB-D347864B0236__B9EFD33D-287E-4624-A89A-A9F990CAC742|2020-09-30    |E1F74FBC-2A2E-409F-B9A2-97E18986AF73|2020-09-22 00:44:41|228070.0       |AIR FILTERS      |
|00003D26-F6BB-457D-9CCB-D347864B0236__B9EFD33D-287E-4624-A89A-A9F990CAC742|2020-09-30    |E1F74FBC-2A2E-409F-B9A2-97E18986AF73|2020-09-22 00:44:41|228070.0       |CABIN AIR FILTERS|
|00003D26-F6BB-457D-9CCB-D347864B0236__B9EFD33D-287E-4624-A89A-A9F990CAC742|2020-09-3

In [16]:
base_sales.filter(
    f.col("transaction_id").isNotNull()
).groupBy("product_category", "_id").agg(
    f.countDistinct("transaction_id").alias("count")
).filter(
    f.col("count") > 5
).show(5, truncate=False)

+----------------+--------------------------------------------------------------------------+-----+
|product_category|_id                                                                       |count|
+----------------+--------------------------------------------------------------------------+-----+
|OIL FILTERS     |ADF9EA7B-950A-4DF8-B3E4-FA44F535916A__E89BEB7A-E3C4-49B1-B137-5347EE53626D|13   |
|OIL             |19D9BC46-B71F-46E1-8A7D-5A4F48AAE2FF__6B23EF10-4823-4457-A516-E3A69F2139BF|40   |
|OIL             |7AE928E6-4908-4E9A-9B83-93B773ABC398__92F5C240-D131-43FF-B705-024580651AF3|6    |
|SERVICE         |B38124C6-B94C-4B39-8DEB-BAF6113B0348__76FC4BAB-928B-487A-92B9-FC06C7F48739|7    |
|OIL SYNTHETIC   |884884AE-CAAF-4257-B425-13E8FF898B6F__F0BA9438-43A9-457A-9E83-25D9443362CD|12   |
+----------------+--------------------------------------------------------------------------+-----+
only showing top 5 rows



In [ ]:
base_sales.filter(
    f.col("transaction_id").isNotNull()
).withColumn(
    "major_category",
    f.when(
        f.lower("product_category").isin("oil"),
        "oil"
    ).when(
        f.lower("product_category").isin("oil synthetic"),
        "oil_synthetic"
    ).when(
        f.contains(f.lower("product_category"), f.lit("oil filter")),
        "oil filter"
    ).when(
        f.contains(f.lower("product_category"), f.lit("air filter")),
        "air filter"
    ).when(
        f.contains(f.lower("product_category"), f.lit("ac ")),
        "ac"
    ).when(
        f.contains(f.lower("product_category"), f.lit("tyres")),
        "tyres"
    ).when(
        f.contains(f.lower("product_category"), f.lit("batt")),
        "batteries"
    ).when(
        f.contains(f.lower("product_category"), f.lit("transmission")),
        "transmission"
    ).when(
        f.contains(f.lower("product_category"), f.lit("engine")),
        "engine"
    ).when(
        f.contains(f.lower("product_category"), f.lit("additives")),
        "additives"
    ).when(
        f.contains(f.lower("product_category"), f.lit("plug")),
        "plug"
    ).when(
        f.contains(f.lower("product_category"), f.lit("fuel")),
        "fuel"
    ).when(
        f.contains(f.lower("product_category"), f.lit("gear")),
        "gear"
    ).when(
        f.contains(f.lower("product_category"), f.lit("brake")),
        "brake"
    ).when(
        f.contains(f.lower("product_category"), f.lit("coolant")),
        "coolant"
    ).otherwise("others")
).filter(
    f.col("major_category") != "others"
).groupBy("product_category", "_id").agg(
    f.countDistinct("transaction_id").alias("count")
).filter(
    f.col("count") > 5
).groupby(
    "product_category"
).agg(
    f.countDistinct("_id").alias("count")
).orderBy(f.desc("count")).show(50, truncate=False)

+--------------------+------+
|product_category    |count |
+--------------------+------+
|SERVICE             |194865|
|BRAKE FLUID         |15243 |
|COOLANTS            |11760 |
|NISSAN-MISCELLANEOUS|9668  |
|CONSUMABLES         |6556  |
|TOP OFFS            |964   |
|WIPER BLADES        |775   |
|GASKIT              |298   |
|AIR DRYER FILTERS   |18    |
|HEADLIGHT SERVICE   |10    |
|DEACTIVE ITEM       |9     |
|OBS                 |1     |
|WATER FILTERS       |1     |
+--------------------+------+



In [23]:
ftr_trx_mileage = base_sales.groupBy(
    "_id", "_observ_end_dt", "transaction_id", "transaction_dt"
).agg(
    f.mean("current_mileage").alias("current_mileage"),
    f.mean("previous_mileage").alias("previous_mileage"),
).withColumn(
    "current_mileage",
    f.when(
        (f.abs(f.col("current_mileage") - f.avg("current_mileage").over(win_id_last_3)) > f.std("current_mileage").over(win_id_all)) & (f.avg("current_mileage").over(win_id_last_3).isNotNull()),
        None
    ).when(
        (f.abs(f.col("current_mileage") - f.avg("current_mileage").over(win_id_next_3)) > f.std("current_mileage").over(win_id_all)) & (f.avg("current_mileage").over(win_id_next_3).isNotNull()),
        None
    ).otherwise(f.col("current_mileage"))
).withColumn(
    "current_mileage",
    f.when(
        (
            (f.col("current_mileage") - f.last("current_mileage").over(win_id)) / 
            f.when(f.date_diff(f.col("transaction_dt"), f.last("transaction_dt").over(win_id)) == 0, 1).otherwise(f.date_diff(f.col("transaction_dt"), f.last("transaction_dt").over(win_id)))
        ) > 1200,
        None
    ).otherwise(f.col("current_mileage"))
).withColumn(
    "transaction_dt",
    f.when(
        f.col("current_mileage").isNull(), None
    ).otherwise(f.col("transaction_dt"))
).withColumn(
    "last_mileage",
    f.last("current_mileage", ignorenulls=True).over(win_id)
).withColumn(
    "last_transaction_dt",
    f.last("transaction_dt", ignorenulls=True).over(win_id)
).withColumn(
    "run_mileage",
    f.col("current_mileage") - f.col("last_mileage")
).withColumn(
    "increased_period",
    f.date_diff(f.col("transaction_dt"), f.col("last_transaction_dt"))
).withColumn(
    "avg_mileage_per_day",
    f.col("run_mileage") / f.col("increased_period")
)

In [25]:
ftr_trx_mileage.filter(
    f.col("_id") == working_ids[5]
).orderBy("_id", "_observ_end_dt", "transaction_dt").show(40, truncate=False)

+--------------------------------------------------------------------------+--------------+------------------------------------+-------------------+---------------+----------------+------------+-------------------+-----------+----------------+-------------------+
|_id                                                                       |_observ_end_dt|transaction_id                      |transaction_dt     |current_mileage|previous_mileage|last_mileage|last_transaction_dt|run_mileage|increased_period|avg_mileage_per_day|
+--------------------------------------------------------------------------+--------------+------------------------------------+-------------------+---------------+----------------+------------+-------------------+-----------+----------------+-------------------+
|A4F52279-0F64-4165-BB89-600CD3314D59__4D5A9841-55D7-492F-B695-78DE31B6A760|2021-11-30    |FABF768E-CA0B-4210-928B-97913E1D5815|2021-11-30 15:15:25|89099.0        |78724.0         |NULL        |NULL          

In [58]:
ftr_unit_mileage = ftr_trx_mileage.groupBy(
    "_id", "_observ_end_dt",
).agg(
    f.last("transaction_dt", ignorenulls=True).alias("last_transaction_dt"),
    f.min("current_mileage").alias("month_min_current_mileage"),
    f.max("current_mileage").alias("month_max_current_mileage"),
    f.mean("run_mileage").alias("month_avg_run_mileage"),
    f.mean("increased_period").alias("month_avg_run_period"),
    f.mean("avg_mileage_per_day").alias("month_avg_mileage_per_day"),
).withColumn(
    "is_single_visit",
    f.when(
        (f.col("month_max_current_mileage").isNull() |
        f.col("month_min_current_mileage").isNull()),
        None
    ).when(
        f.col("month_max_current_mileage") == f.col("month_min_current_mileage"),
        1
    ).otherwise(0)
).withColumn(
    "is_multiple_visit",
    f.when(
        (f.col("month_max_current_mileage").isNull() |
        f.col("month_min_current_mileage").isNull()),
        None
    ).when(
        f.col("month_max_current_mileage") == f.col("month_min_current_mileage"),
        0
    ).otherwise(1)
).withColumn(
    "month_avg_mileage_per_day",
    f.when(
        f.col("month_avg_mileage_per_day").isNull(),
        f.round(f.first("month_avg_mileage_per_day", ignorenulls=True).over(win_id_monthly.rowsBetween(0, Window.unboundedFollowing)), 2)
    ).otherwise(f.col("month_avg_mileage_per_day"))
)

In [52]:
global_avg_mileage = ftr_unit_mileage.select(f.round(f.median("month_avg_mileage_per_day"), 0)).collect()[0][0]


In [53]:
global_avg_mileage

76.0

In [59]:

ftr_unit_mileage = ftr_unit_mileage.withColumn(
    "month_avg_mileage_per_day",
    f.when(
        (f.col("month_min_current_mileage").isNotNull()) &
        ((f.col("month_avg_mileage_per_day") > 1500) | (f.col("month_avg_mileage_per_day").isNull())),
        global_avg_mileage
    ).otherwise(f.col("month_avg_mileage_per_day"))
)

In [60]:
ftr_unit_mileage.filter(
    f.col("_id") == working_ids[5]
).orderBy("_id", "_observ_end_dt",).show(40, truncate=False)

+--------------------------------------------------------------------------+--------------+-------------------+-------------------------+-------------------------+---------------------+--------------------+-------------------------+---------------+-----------------+
|_id                                                                       |_observ_end_dt|last_transaction_dt|month_min_current_mileage|month_max_current_mileage|month_avg_run_mileage|month_avg_run_period|month_avg_mileage_per_day|is_single_visit|is_multiple_visit|
+--------------------------------------------------------------------------+--------------+-------------------+-------------------------+-------------------------+---------------------+--------------------+-------------------------+---------------+-----------------+
|A4F52279-0F64-4165-BB89-600CD3314D59__4D5A9841-55D7-492F-B695-78DE31B6A760|2021-11-30    |2021-11-30 15:15:25|89099.0                  |89099.0                  |NULL                 |NULL          

In [76]:
w3 = Window.partitionBy("_id").orderBy(f.col("_observ_end_dt")).rowsBetween(-3, 0)

ftr_unit_forecast = ftr_unit_mileage.withColumn(
    "days_until_end_next_1_quarters",
    f.date_diff(
        f.trunc(f.add_months("last_transaction_dt", 3), "quarter"),
        f.col("last_transaction_dt")
    )
).withColumn(
    "month_avg_mileage_per_day_nullif_past_3_next_0_quarters",
    f.avg(
        f.replace(f.col("month_avg_mileage_per_day"), f.lit(0.0))
    ).over(w3)
).withColumn(
#     "forecasted_increased_mileage_next_1_quarters",
#     f.col("month_avg_mileage_per_day_nullif_past_3_next_0_quarters") * f.col("days_until_end_next_1_quarters")
# ).withColumn(
#     "is_forecasted_mileage_next_1_quarters_above_syn_threshold",
#     f.when(f.col("forecasted_increased_mileage_next_1_quarters") >= 8000, 1).otherwise(0) # synthetic oil
# ).withColumn(
#     "is_forecasted_mileage_next_1_quarters_above_oil_threshold",
#     f.when(f.col("forecasted_increased_mileage_next_1_quarters") >= 5000, 1).otherwise(0) # mineral oil
# ).withColumn(
    "estimated_days_to_change_mineral_oil",
    f.when(
        f.col("month_min_current_mileage").isNotNull(),
        f.round(f.lit(5000) / f.col("month_avg_mileage_per_day_nullif_past_3_next_0_quarters")) # mineral oil
    ).otherwise(None)
).withColumn(
    "estimated_days_to_change_synthetic_oil",
    f.when(
        f.col("month_min_current_mileage").isNotNull(),
        f.round(f.lit(8000) / f.col("month_avg_mileage_per_day_nullif_past_3_next_0_quarters")) # synthetic oil
    ).otherwise(None)
).withColumn(
    "estimated_days_to_change_mineral_oil",
    f.last("estimated_days_to_change_mineral_oil", ignorenulls=True).over(win_id_monthly.rangeBetween(Window.unboundedPreceding, 0))
).withColumn(
    "estimated_days_to_change_synthetic_oil",
    f.last("estimated_days_to_change_synthetic_oil", ignorenulls=True).over(win_id_monthly.rangeBetween(Window.unboundedPreceding, 0))
).withColumn(
    "last_transaction_dt",
    f.last("last_transaction_dt", ignorenulls=True).over(win_id_monthly.rangeBetween(Window.unboundedPreceding, 0))
).withColumn(
    "days_since_last_transactions",
    f.date_diff("_observ_end_dt", "last_transaction_dt")
).withColumn(
    "below_estimated_mineral_oil_change",
    ((f.col("estimated_days_to_change_mineral_oil") - f.col("days_since_last_transactions")) > 0).cast("int")
).withColumn(
    "below_estimated_synthetic_oil_change",
    ((f.col("estimated_days_to_change_synthetic_oil") - f.col("days_since_last_transactions")) > 0).cast("int")
)

In [77]:

ftr_unit_forecast.filter(
    f.col("_id") == working_ids[5]
).orderBy("_id", "_observ_end_dt").show(50)

+--------------------+--------------+-------------------+-------------------------+-------------------------+---------------------+--------------------+-------------------------+---------------+-----------------+------------------------------+-------------------------------------------------------+------------------------------------+--------------------------------------+----------------------------+----------------------------------+------------------------------------+
|                 _id|_observ_end_dt|last_transaction_dt|month_min_current_mileage|month_max_current_mileage|month_avg_run_mileage|month_avg_run_period|month_avg_mileage_per_day|is_single_visit|is_multiple_visit|days_until_end_next_1_quarters|month_avg_mileage_per_day_nullif_past_3_next_0_quarters|estimated_days_to_change_mineral_oil|estimated_days_to_change_synthetic_oil|days_since_last_transactions|below_estimated_mineral_oil_change|below_estimated_synthetic_oil_change|
+--------------------+--------------+---------

In [225]:
out = base_sales.select(
    "_id", "_observ_end_dt"
).distinct().join(
    ftr_unit_forecast,
    on=["_id", "_observ_end_dt"],
    how="left"
)

In [226]:
print(out.count())
print(base_sales.select("_id", "_observ_end_dt").distinct().count())

44111706


44111706


# Churn features

In [ ]:
w_churn = Window.partitionBy("_id").orderBy(f.col("_observ_end_dt")).rowsBetween(1, 1)
w_churn_2 = Window.partitionBy("_id").orderBy(f.col("_observ_end_dt")).rowsBetween(1, 2)


In [ ]:
# out = features.withColumn(
#     "customer_mineral_oil",
#     f.when(
#         f.col("has_oil") > 0,
#         1
#     ).otherwise(0)
# ).withColumn(
#     "customer_synthetic_oil",
#     f.when(
#         f.col("has_oil_synthetic") > 0,
#         1
#     ).otherwise(0)
# ).withColumn(
#     "target_churn",
#     f.when(
#         (f.col("customer_mineral_oil") > 0) & 
#         (f.col("is_forecasted_mileage_next_1_quarters_above_oil_threshold") == True) &
#         (f.sum("is_active").over(w_churn) > 0),
#         0
#     ).when(
#         (f.col("customer_synthetic_oil") > 0) & 
#         (f.col("is_forecasted_mileage_next_2_quarter_above_syn_threshold") == True) &
#         (f.sum("is_active").over(w_churn_2) > 0),
#         0
#     ).otherwise(1)
# )

# Result

In [7]:
# prm_transactions = spark.read.parquet("../data/03_primary/prm_transactions")
ftr_transactions = spark.read.parquet("../data/04_feature/ftr_transactions")


In [5]:
prm_transactions.columns

['_id',
 '_observ_end_dt',
 'quarter_total_sales',
 'quarter_net_sales',
 'quarter_total_profit',
 'quarter_total_cost',
 'quarter_total_amount_discounts',
 'quarter_total_qty_discounts',
 'quarter_distinct_skus_sold',
 'quarter_distinct_branches',
 'quarter_distinct_product_categories',
 'quarter_distinct_transactions',
 'quarter_avg_order',
 'quarter_avg_skus_per_order',
 'quarter_net_sales_category_ac',
 'quarter_total_profit_category_ac',
 'quarter_total_cost_category_ac',
 'quarter_total_amount_discounts_category_ac',
 'quarter_total_qty_discounts_category_ac',
 'quarter_net_sales_category_additives',
 'quarter_total_profit_category_additives',
 'quarter_total_cost_category_additives',
 'quarter_total_amount_discounts_category_additives',
 'quarter_total_qty_discounts_category_additives',
 'quarter_net_sales_category_air_filter',
 'quarter_total_profit_category_air_filter',
 'quarter_total_cost_category_air_filter',
 'quarter_total_amount_discounts_category_air_filter',
 'quarter_

25/04/30 02:01:19 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
ftr_transactions.columns

['_id',
 '_observ_end_dt',
 'quarter_total_sales',
 'quarter_net_sales',
 'quarter_total_profit',
 'quarter_total_cost',
 'quarter_total_amount_discounts',
 'quarter_total_qty_discounts',
 'quarter_distinct_skus_sold',
 'quarter_distinct_branches',
 'quarter_distinct_product_categories',
 'quarter_distinct_transactions',
 'quarter_avg_order',
 'quarter_avg_skus_per_order',
 'quarter_net_sales_category_ac',
 'quarter_total_profit_category_ac',
 'quarter_total_cost_category_ac',
 'quarter_total_amount_discounts_category_ac',
 'quarter_total_qty_discounts_category_ac',
 'quarter_net_sales_category_additives',
 'quarter_total_profit_category_additives',
 'quarter_total_cost_category_additives',
 'quarter_total_amount_discounts_category_additives',
 'quarter_total_qty_discounts_category_additives',
 'quarter_net_sales_category_air_filter',
 'quarter_total_profit_category_air_filter',
 'quarter_total_cost_category_air_filter',
 'quarter_total_amount_discounts_category_air_filter',
 'quarter_

25/04/30 03:47:57 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
25/04/30 10:02:01 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1027229 ms exceeds timeout 120000 ms
25/04/30 10:02:01 WARN SparkContext: Killing executors is not supported by current scheduler.
25/04/30 10:05:59 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.Rp

In [ ]:
x = set([
    
    
])
print(len(x))

sorted([col for col in x])


14


['avg_pop_density',
 'customer_synthetic_oil',
 'is_active_sum_past_7_next_0_quarters',
 'is_single_visit',
 'last_trx_current_mileage',
 'last_trx_net_sales',
 'last_trx_total_amount_discounts',
 'most_frequent_day_of_month_mean_past_3_next_0_quarters',
 'most_frequent_day_of_week_mean_past_3_next_0_quarters',
 'most_trx_branch_n_places_within_0300m__All',
 'most_trx_branch_pop_density',
 'quarter_avg_order_mean_past_3_next_0_quarters',
 'quarter_distinct_product_categories_mean_past_3_next_0_quarters',
 'quarter_distinct_skus_sold_mean_past_3_next_0_quarters']

In [ ]:
feat_list = [
    "most_trx_branch_n_places_within_0300m__All",
    "most_trx_branch_pop_density",
    "avg_n_places_within_0300m__All",
    "avg_pop_density",

    "most_frequent_day_of_month",
    "most_frequent_day_of_month_mean_past_3_next_0_quarters",
    "most_frequent_day_of_week",
    "most_frequent_day_of_week_mean_past_3_next_0_quarters",

    "last_trx_net_sales",
    "last_trx_total_amount_discounts",
    "last_trx_current_mileage",

    "customer_synthetic_oil",
    "quarter_avg_mileage_per_day_nullif_past_3_next_0_quarters",
    "quarter_avg_order_mean_past_3_next_0_quarters",
    "is_single_visit",
    "is_active_sum_past_7_next_0_quarters"
]